In [26]:
%pip install transformers datasets accelerate bitsandbytes peft trl torch -q

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [52]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig, set_seed

model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [53]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer ready: {len(tokenizer)} tokens")

✅ Tokenizer ready: 128256 tokens


In [54]:
# Add special tokens for function calling
special_tokens = ["<think>", "</think>", "<function>", "</function>"]

# Add to tokenizer
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})

print(f"Added special tokens: {special_tokens}")
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Test tokenization of special tokens
test_text = "<think>This is a test</think>"
tokens = tokenizer.tokenize(test_text)
print(f"Tokenized '<function>This is a test</function>': {tokens}")

Added special tokens: ['<think>', '</think>', '<function>', '</function>']
Tokenizer vocab size: 128260
Tokenized '<function>This is a test</function>': ['<think>', 'This', 'Ġis', 'Ġa', 'Ġtest', '</think>']


In [ ]:
from datasets import load_from_disk
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset"
dataset = load_from_disk(f"{dataset_based_path}/mh-function-calling-dataset-hf")

In [56]:
dataset

DatasetDict({
    train: Dataset({
        features: ['patient_query', 'assistant_response', 'function_name', 'patient_id', 'num_days', 'text'],
        num_rows: 3200
    })
    test: Dataset({
        features: ['patient_query', 'assistant_response', 'function_name', 'patient_id', 'num_days', 'text'],
        num_rows: 800
    })
})

In [57]:
# Load the model with full precisoin
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,  # Full precision
    attn_implementation='eager',
    device_map={"": 0}           # Choose the first GPU
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.58it/s]


In [61]:
# Update the tokens embedding with the new four special token
model.resize_token_embeddings(len(tokenizer))
print(f"Model embeddings resized to: {model.get_input_embeddings().num_embeddings}")

Model embeddings resized to: 128260


#### Configure the LoRA

In [62]:
from peft import LoraConfig, TaskType

rank_dimension = 16  # Good balance for 3B model
lora_alpha = 64      # Lower than their 64 - more conservative for function calling
lora_dropout = 0.05   # Slightly higher to prevent overfitting on your smaller dataset

peft_config = LoraConfig(
    r=rank_dimension,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers (most important)
        "gate_proj", "up_proj", "down_proj",     # MLP layers
        "embed_tokens", "lm_head"                # Need for stronger adaptation
    ],
    task_type=TaskType.CAUSAL_LM,
    # bias="none"  # Add this for better performance
)

In [63]:
from trl import SFTTrainer, SFTConfig

# Configure training arguments for mental health function calling
training_args = SFTConfig(
    output_dir="/home/jovyan/Works/MSc_Thesis/mh_function_calling/model_training/finetuned_models/llama3.2_3b_mh_function_calling",
    
    # Batch configuration - small batches with accumulation for memory efficiency
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,  # Effective batch size = 4
    
    # Learning configuration optimized for function calling
    learning_rate=1e-4,  # Conservative rate for stable training
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    weight_decay=0.1,
    
    # Training duration - more epochs for smaller specialized dataset
    num_train_epochs=5,
    
    # Evaluation and monitoring
    eval_strategy="epoch",
    logging_steps=5,
    save_strategy="no",  # Only save final model
    
    # Memory optimization for 3B model
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    
    # Sequence handling for mental health conversations
    max_seq_length=1024,  # Adequate for conversation + function calls
    packing=True,  # Efficient training on variable length sequences
    # dataset_text_field="text",
    
    push_to_hub=False,
)

print(f"📊 Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")


📊 Effective batch size: 4


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    peft_config=peft_config,
)

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:190: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:574: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(
/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/trl/trainer/sft_trainer.py:412: UserWarning: Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your att

In [66]:
print("🚀 Starting training...")
trainer.train()

🚀 Starting training...


Epoch,Training Loss,Validation Loss
1,0.190200,0.176837
2,0.137500,0.132096
3,0.091400,0.092378
4,0.088800,0.090478
5,0.084600,0.090417


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=365, training_loss=0.2457376774043253, metrics={'train_runtime': 749.7256, 'train_samples_per_second': 3.861, 'train_steps_per_second': 0.487, 'total_flos': 4.691766964503552e+16, 'train_loss': 0.2457376774043253})

In [67]:
# Save the final model
print("Saving model...")
trainer.save_model()
tokenizer.save_pretrained(training_args.output_dir)

print("Training complete!")
print(f"Model saved to: {training_args.output_dir}")

Saving model...


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:222: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Training complete!
Model saved to: /home/jovyan/Works/MSc_Thesis/mh_function_calling/model_training/finetuned_models/llama3.2_3b_mh_function_calling
